# PASTA 2.0 — Best Energy & % β-Strand  ·  Aβ42 Variants

| Metric | Destabilizing mutation   |
|------------|---------------------|
| `Best Energy` | **ΔEnergy > 0** — closer to zero → less stable amyloid |
| `% β-Strand`  | **Δβ < 0** — fewer β-strands → lower propensity for aggregation |

In [ ]:
import io
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

BG = "#0a0c14"
SURFACE = "#111520"
MUTED = "#64748b"
TEXT = "#e2e8f0"
BORDER = "#1e2540"
WT_COLOR = "#4ff7c0"
C_DESTAB = "#3b82f6"  # destabilizing — blue
C_NEUTRAL = "#64748b"  # neutral — gray
C_STAB = "#ef4444"  # stabilizing — red

# ── Load data ─────────────────────────────────────────────────
with open("../predictions/pasta/pasta.csv", encoding="utf-8-sig") as f:
    content = f.read().replace(",", ".")

df = pd.read_csv(io.StringIO(content), sep=";")
df["short"] = df["Protein name"].str.replace(r"_Abeta_?42$", "", regex=True)

WT = df[df["Protein name"] == "Wildtype_Abeta_42"].iloc[0]
df["ΔEnergy"] = df["Best Energy"] - WT["Best Energy"]
df["ΔβStrand"] = df["% β-Strand"] - WT["% β-Strand"]

muts = df[df["Protein name"] != "Wildtype_Abeta_42"].copy()


def ecolor(d):  # by ΔEnergy
    if d > 0.1:
        return C_DESTAB
    elif d < -0.1:
        return C_STAB
    return C_NEUTRAL


def bcolor(d):  # by ΔβStrand
    if d < -0.1:
        return C_DESTAB
    elif d > 0.1:
        return C_STAB
    return C_NEUTRAL


print(f"Wildtype Best Energy : {WT['Best Energy']:.4f} kcal/mol")
print(f"Wildtype % β-Strand  : {WT['% β-Strand']:.2f}%")
print()
print("ΔEnergy > 0 (energy‑destabilizing) :", (muts["ΔEnergy"] > 0.1).sum())
print("ΔβStrand < 0 (β‑strand‑destabilizing):", (muts["ΔβStrand"] < -0.1).sum())
print()
print("Top 5 energy‑destabilizing (largest ΔEnergy > 0):")
print(
    muts.nlargest(5, "ΔEnergy")[["short", "Best Energy", "ΔEnergy"]].to_string(
        index=False
    )
)
print()
print("Top 5 β‑strand‑destabilizing (smallest ΔβStrand < 0):")
print(
    muts.nsmallest(5, "ΔβStrand")[["short", "% β-Strand", "ΔβStrand"]].to_string(
        index=False
    )
)

In [ ]:
# Plot 1 — ΔBest Energy lollipop

sv = muts.sort_values("ΔEnergy", ascending=False)
n = len(sv)

fig, ax = plt.subplots(figsize=(12, n * 0.28 + 2), facecolor=BG)
ax.set_facecolor(SURFACE)

for i in range(n):
    if i % 2 == 0:
        ax.axhspan(i - 0.5, i + 0.5, color="white", alpha=0.018, zorder=0)

ax.axvline(0, color=WT_COLOR, lw=1.0, linestyle="--", alpha=0.55)
ax.text(
    0,
    n + 0.15,
    "WT",
    ha="center",
    va="bottom",
    color=WT_COLOR,
    fontsize=8,
    fontfamily="monospace",
    fontweight="bold",
)

# Zones
ax.axvspan(0.1, 2.3, color=C_DESTAB, alpha=0.055)
ax.axvspan(-3.8, -0.1, color=C_STAB, alpha=0.04)
ax.axvline(0.1, color="white", lw=0.4, alpha=0.12)

for i, (_, row) in enumerate(sv.iterrows()):
    c = ecolor(row["ΔEnergy"])
    ax.plot([0, row["ΔEnergy"]], [i, i], color=c, lw=0.9, alpha=0.5)
    ax.scatter(row["ΔEnergy"], i, color=c, s=42, zorder=3, linewidths=0)

ax.set_yticks(range(n))
ax.set_yticklabels(sv["short"].values, fontsize=6.5, fontfamily="monospace")
for tick, (_, row) in zip(ax.get_yticklabels(), sv.iterrows()):
    tick.set_color(ecolor(row["ΔEnergy"]))
    tick.set_alpha(0.88)

ax.tick_params(axis="y", length=0, pad=4)
ax.tick_params(axis="x", colors=MUTED, labelsize=7)
for sp in ax.spines.values():
    sp.set_visible(False)
ax.set_xlim(-3.8, 2.3)
ax.set_ylim(-1, n)

ax.set_xlabel(
    "ΔBest Energy  (mutant − WT,  kcal/mol)\n"
    "> 0  →  closer to zero  →  destabilizing mutation",
    color=MUTED,
    fontsize=8.5,
    fontfamily="monospace",
    labelpad=10,
)

legend_patches = [
    mpatches.Patch(
        facecolor=C_DESTAB,
        label=f"Destabilizing  (Δ > 0)  · n={(sv['ΔEnergy'] > 0.1).sum()}",
    ),
    mpatches.Patch(
        facecolor=C_NEUTRAL,
        label=f"Neutral  (|Δ| ≤ 0.1)  · n={(sv['ΔEnergy'].abs() <= 0.1).sum()}",
    ),
    mpatches.Patch(
        facecolor=C_STAB,
        label=f"Stabilizing  (Δ < 0)  · n={(sv['ΔEnergy'] < -0.1).sum()}",
    ),
]
ax.legend(
    handles=legend_patches,
    loc="lower right",
    frameon=True,
    framealpha=0.15,
    edgecolor=MUTED,
    facecolor=SURFACE,
    fontsize=7.5,
    labelcolor=TEXT,
)

ax.set_title(
    f"PASTA 2.0  ·  ΔBest Energy vs Wildtype Aβ42\nWT = {WT['Best Energy']:.3f} kcal/mol",
    color="white",
    fontsize=12,
    fontfamily="monospace",
    fontweight="bold",
    pad=12,
)

plt.tight_layout()
plt.savefig(
    "pasta_energy.png", dpi=180, bbox_inches="tight", facecolor=BG, edgecolor="none"
)
print("Saved to pasta_energy.png")
plt.show()

In [ ]:
# Plot 2 — Δ% β-Strand lollipop

sb = muts.sort_values("ΔβStrand", ascending=True)  # destabilizing on top
n2 = len(sb)

fig, ax = plt.subplots(figsize=(12, n2 * 0.28 + 2), facecolor=BG)
ax.set_facecolor(SURFACE)

for i in range(n2):
    if i % 2 == 0:
        ax.axhspan(i - 0.5, i + 0.5, color="white", alpha=0.018, zorder=0)

ax.axvline(0, color=WT_COLOR, lw=1.0, linestyle="--", alpha=0.55)
ax.text(
    0,
    n2 + 0.15,
    "WT",
    ha="center",
    va="bottom",
    color=WT_COLOR,
    fontsize=8,
    fontfamily="monospace",
    fontweight="bold",
)

# Zones
ax.axvspan(-11, -0.1, color=C_DESTAB, alpha=0.055)
ax.axvspan(0.1, 7, color=C_STAB, alpha=0.04)

for i, (_, row) in enumerate(sb.iterrows()):
    c = bcolor(row["ΔβStrand"])
    ax.plot([0, row["ΔβStrand"]], [i, i], color=c, lw=0.9, alpha=0.5)
    ax.scatter(row["ΔβStrand"], i, color=c, s=42, zorder=3, linewidths=0)

ax.set_yticks(range(n2))
ax.set_yticklabels(sb["short"].values, fontsize=6.5, fontfamily="monospace")
for tick, (_, row) in zip(ax.get_yticklabels(), sb.iterrows()):
    tick.set_color(bcolor(row["ΔβStrand"]))
    tick.set_alpha(0.88)

ax.tick_params(axis="y", length=0, pad=4)
ax.tick_params(axis="x", colors=MUTED, labelsize=7)
for sp in ax.spines.values():
    sp.set_visible(False)
ax.set_xlim(-11, 7)
ax.set_ylim(-1, n2)

ax.set_xlabel(
    "Δ% β-Strand  (mutant − WT)\n< 0  →  fewer β-sheets  →  destabilizing mutation",
    color=MUTED,
    fontsize=8.5,
    fontfamily="monospace",
    labelpad=10,
)

n_d = (sb["ΔβStrand"] < -0.1).sum()
n_n = (sb["ΔβStrand"].abs() <= 0.1).sum()
n_s = (sb["ΔβStrand"] > 0.1).sum()
legend_patches2 = [
    mpatches.Patch(facecolor=C_DESTAB, label=f"Destabilizing  (Δ < 0)  · n={n_d}"),
    mpatches.Patch(facecolor=C_NEUTRAL, label=f"Neutral  (Δ = 0)  · n={n_n}"),
    mpatches.Patch(facecolor=C_STAB, label=f"Enhancing  (Δ > 0)  · n={n_s}"),
]
ax.legend(
    handles=legend_patches2,
    loc="lower right",
    frameon=True,
    framealpha=0.15,
    edgecolor=MUTED,
    facecolor=SURFACE,
    fontsize=7.5,
    labelcolor=TEXT,
)

ax.set_title(
    f"PASTA 2.0  ·  Δ% β-Strand vs Wildtype Aβ42\nWT = {WT['% β-Strand']:.2f}%",
    color="white",
    fontsize=12,
    fontfamily="monospace",
    fontweight="bold",
    pad=12,
)

plt.tight_layout()
plt.savefig(
    "pasta_beta.png", dpi=180, bbox_inches="tight", facecolor=BG, edgecolor="none"
)
print("Saved to pasta_beta.png")
plt.show()

In [ ]:
# Plot 3 — Absolute values (both metrics side by side)

sa = df.sort_values("Best Energy", ascending=False)  # worst (> WT) on top

fig, axes = plt.subplots(
    1, 2, figsize=(18, len(sa) * 0.25 + 2.5), facecolor=BG, gridspec_kw={"wspace": 0.04}
)

panels = [
    (
        "Best Energy",
        WT["Best Energy"],
        "Best Energy (kcal/mol)",
        "Best Energy\n(more negative → more stable amyloid)",
    ),
    (
        "% β-Strand",
        WT["% β-Strand"],
        "% β-Strand",
        "% β-Strand\n(higher → more β-sheets → stronger amyloid)",
    ),
]

for ax_i, (col, wt_val, xlabel, title_str) in enumerate(panels):
    ax = axes[ax_i]
    ax.set_facecolor(SURFACE)

    # Left edge of the bar
    left = sa["Best Energy"].min() - 0.3 if col == "Best Energy" else 0

    for i, (_, row) in enumerate(sa.iterrows()):
        is_wt = row["Protein name"] == "Wildtype_Abeta_42"
        if is_wt:
            c = WT_COLOR
        elif col == "Best Energy":
            c = ecolor(row["ΔEnergy"])
        else:
            c = bcolor(row["ΔβStrand"])

        if i % 2 == 0:
            ax.axhspan(i - 0.5, i + 0.5, color="white", alpha=0.018, zorder=0)

        ax.barh(
            i,
            row[col] - left,
            left=left,
            color=c,
            alpha=1.0 if is_wt else 0.75,
            height=0.6,
            zorder=2,
        )

        # Numeric labels
        offset = 0.05 if col == "% β-Strand" else 0.03
        label = f"{row[col]:.2f}" if col == "Best Energy" else f"{row[col]:.1f}%"
        ax.text(
            row[col] + offset,
            i,
            label,
            va="center",
            ha="left",
            color=c,
            fontsize=5.5,
            fontfamily="monospace",
            alpha=0.85,
        )

    # WT line
    ax.axvline(wt_val, color=WT_COLOR, lw=1.0, linestyle="--", alpha=0.55)
    wt_label = f"{wt_val:.3f}" if col == "Best Energy" else f"{wt_val:.2f}%"
    ax.text(
        wt_val,
        len(sa) + 0.1,
        f"WT\n{wt_label}",
        ha="center",
        va="bottom",
        color=WT_COLOR,
        fontsize=7,
        fontfamily="monospace",
        fontweight="bold",
    )

    # Names only on the left
    ax.set_yticks(range(len(sa)))
    if ax_i == 0:
        ax.set_yticklabels(sa["short"].values, fontsize=6.5, fontfamily="monospace")
        for tick, (_, row) in zip(ax.get_yticklabels(), sa.iterrows()):
            is_wt = row["Protein name"] == "Wildtype_Abeta_42"
            tick.set_color(WT_COLOR if is_wt else ecolor(row["ΔEnergy"]))
            tick.set_alpha(1.0 if is_wt else 0.85)
    else:
        ax.set_yticklabels([""] * len(sa))

    ax.tick_params(axis="y", length=0, pad=4)
    ax.tick_params(axis="x", colors=MUTED, labelsize=7)
    for sp in ax.spines.values():
        sp.set_visible(False)
    ax.set_ylim(-1, len(sa))
    ax.set_xlabel(xlabel, color=MUTED, fontsize=9, fontfamily="monospace", labelpad=8)
    ax.set_title(
        title_str,
        color="white",
        fontsize=10,
        fontfamily="monospace",
        fontweight="bold",
        pad=10,
    )

fig.suptitle(
    "PASTA 2.0  ·  Absolute values — all Aβ42 variants\n"
    "Sorted from least to most amyloidogenic",
    color="white",
    fontsize=12,
    fontfamily="monospace",
    fontweight="bold",
    y=1.01,
)

lp = [
    mpatches.Patch(facecolor=WT_COLOR, label="Wildtype"),
    mpatches.Patch(facecolor=C_DESTAB, label="Destabilizing mutation"),
    mpatches.Patch(facecolor=C_NEUTRAL, label="Neutral"),
    mpatches.Patch(facecolor=C_STAB, label="Stabilizing"),
]
fig.legend(
    handles=lp,
    loc="lower center",
    ncol=4,
    frameon=False,
    fontsize=8,
    labelcolor=TEXT,
    bbox_to_anchor=(0.5, -0.01),
)

plt.tight_layout()
plt.savefig(
    "pasta_absolute.png", dpi=180, bbox_inches="tight", facecolor=BG, edgecolor="none"
)
print("Saved to pasta_absolute.png")
plt.show()